# R2AI Financial QA — chạy trên Kaggle

Pipeline hỏi–đáp số liệu báo cáo tài chính tiếng Việt.

**Chuẩn bị trước khi chạy** (xem `notebooks/README.md` để biết chi tiết):

1. Tạo Kaggle Dataset chứa `retrieval_documents.jsonl`, `data/dictionaries/`, `data/questions/`
2. Add dataset đó vào notebook (nút **Add Input**)
3. Bật **Internet** trong Settings (để cài thư viện + tải model)
4. Chọn **Accelerator = GPU T4** nếu muốn dùng LLM


## 1. Lấy mã nguồn


In [ ]:
# Cách A — clone từ GitHub (cần bật Internet)
!git clone -q https://github.com/<user>/r2ai-doublem-chatbotAI.git /kaggle/working/r2ai || echo 'đã có sẵn'

# Cách B — nếu bạn upload mã nguồn thành một dataset thì copy sang thư mục ghi được:
# !cp -r /kaggle/input/<ten-dataset-ma-nguon> /kaggle/working/r2ai

import sys
sys.path.insert(0, '/kaggle/working/r2ai')


## 2. Cài thư viện, dò dữ liệu, build BM25 index

`setup()` tự tìm dữ liệu trong `/kaggle/input/**` nên không cần sửa đường dẫn.
Index BM25 (~560 MB) được build từ `retrieval_documents.jsonl` trong khoảng 2–3 phút
và lưu vào `/kaggle/working/indexes/` — **không cần upload index**.


In [ ]:
from src.kaggle_setup import setup

paths = setup()


## 3. Dựng pipeline

| `llm_backend` | Khi nào dùng |
|---|---|
| `'none'` | Nhanh nhất, không cần GPU. Chỉ trả lời bằng đường tra cứu tất định — kết quả ổn định, lặp lại được |
| `'hf'` | Nạp Qwen2.5-3B-Instruct bằng transformers. Cần GPU T4 + Internet |
| `'auto'` | Dò Ollama trước, không có thì dùng `hf`, vẫn không được thì `none` |


In [ ]:
from src.kaggle_setup import build_pipeline

# Bắt đầu bằng 'none' để kiểm tra dữ liệu đã đúng chưa (chạy trong vài giây)
pipeline = build_pipeline(llm_backend='none', use_llm_router=False, paths=paths)

result = pipeline.run('Chi phí phạt của công ty mẹ SCR năm 2017 là bao nhiêu tỷ đồng?')
print(result['answer'])
print(result['citations'])


## 4. (Tuỳ chọn) Bật LLM để xử lý câu hỏi cần suy luận

Cần Accelerator = GPU T4. Lần đầu sẽ tải model (~6 GB).


In [ ]:
# pipeline = build_pipeline(llm_backend='hf', paths=paths)
# print(pipeline.run('Chi phí lương và các khoản khác theo lương của công ty mẹ FTS năm 2021?')['answer'])


## 5. Chạy toàn bộ bộ câu hỏi

`--resume` cho phép chạy tiếp khi notebook bị hết giờ (Kaggle giới hạn 9–12 tiếng/phiên).


In [ ]:
!cd /kaggle/working/r2ai && python run_all_questions.py \
    --llm-backend none \
    --no-router \
    --limit 50 \
    --output /kaggle/working/results_all.jsonl \
    --resume


## 6. Xem kết quả


In [ ]:
import json, pandas as pd

rows = [json.loads(l) for l in open('/kaggle/working/results_all.jsonl') if l.strip()]
df = pd.DataFrame(rows).drop_duplicates('id', keep='last')

print(f"Trả lời được: {df['success'].sum()}/{len(df)}  |  thời gian TB: {df['duration_s'].mean():.1f}s")
df[['id', 'question', 'answer', 'success', 'duration_s']].head(20)


---
### Lưu ý

- Câu trả lời có dạng `X đơn vị (giá trị gốc: ...)` đến từ **đường tra cứu tất định** — ổn định giữa các lần chạy.
- Câu trả lời dạng câu văn đến từ **code pandas do LLM sinh** — có thể khác nhau giữa các lần chạy, cần đối chiếu lại.
- Hệ thống chủ động **trả lời “không tìm thấy”** khi bằng chứng không rõ ràng, thay vì đoán một con số.
